Import packages 

In [22]:
import duckdb
import pyarrow.parquet as pq
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")  # .env is in data_concierge (parent of retrieval)

DATASET_DIR = "dataset"  # or "retrieval/dataset" if running from project root
CSV_OUTPUT_DIR = "csv_files"
KEY_NAME = os.getenv("PARQUET_KEY_NAME")
KEY_VALUE = os.getenv("PARQUET_KEY_VALUE")

Exporting all parquet file as a csv file

In [23]:
os.makedirs(CSV_OUTPUT_DIR, exist_ok=True)

con = duckdb.connect()
con.execute(f"PRAGMA add_parquet_key('{KEY_NAME}', '{KEY_VALUE}');")

for f in os.listdir(DATASET_DIR):
    if f.endswith(".parquet"):
        parquet_path = os.path.join(DATASET_DIR, f)
        csv_path = os.path.join(CSV_OUTPUT_DIR, f.replace(".parquet", ".csv"))
        con.execute(f"""
            COPY (
                SELECT * FROM read_parquet(
                    '{parquet_path}',
                    encryption_config = {{footer_key: '{KEY_NAME}'}}
                )
            ) TO '{csv_path}' (HEADER, DELIMITER ',');
        """)
        print(f"Exported {f} -> {csv_path}")

Exported drug_exposure_cancerdrugs.parquet -> csv_files/drug_exposure_cancerdrugs.csv
Exported measurement_mutation.parquet -> csv_files/measurement_mutation.csv
Exported person.parquet -> csv_files/person.csv
Exported condition_occurrence.parquet -> csv_files/condition_occurrence.csv
Exported procedure_occurrence.parquet -> csv_files/procedure_occurrence.csv
Exported death.parquet -> csv_files/death.csv
